In [62]:
import pandas as pd
import numpy as np

df=pd.read_excel("../data/cleandata.xlsx")
df.head()
#Did I load the dataset correctly?




,date,Power Consumption,Outdoor Temperature,Occupancy
0,2018-05-22 00:00:00,71.98,15.72,1
1,2018-05-22 00:15:00,71.00,15.37,1
2,2018-05-22 00:30:00,65.20,15.27,0
3,2018-05-22 00:45:00,95.90,16.03,0
4,2018-05-22 01:00:00,43.60,15.28,0


In [4]:
print(df.columns.tolist())

['date', 'Power Consumption', 'Outdoor Temperature', 'Occupancy']


In [5]:
df.head(10)

,date,Power Consumption,Outdoor Temperature,Occupancy
0,2018-05-22 00:00:00,71.980000,15.72,1
1,2018-05-22 00:15:00,71.000000,15.37,1
2,2018-05-22 00:30:00,65.200000,15.27,0
3,2018-05-22 00:45:00,95.900000,16.03,0
4,2018-05-22 01:00:00,43.600000,15.28,0
5,2018-05-22 01:15:00,40.500000,15.12,0
6,2018-05-22 01:30:00,50.389999,14.23,0
7,2018-05-22 01:45:00,74.390000,13.62,0
8,2018-05-22 02:00:00,31.775000,13.41,0
9,2018-05-22 02:15:00,48.350000,12.51,0


In [6]:
df["Power_Norm"] = (
    df["Power Consumption"]
    /
    df["Power Consumption"].max()
)

df["Occupancy_Norm"] = (
    df["Occupancy"]
    /
    df["Occupancy"].max()
)

df["Temperature_Norm"] = (
    df["Outdoor Temperature"]
    /
    df["Outdoor Temperature"].max()
)

In [7]:
print(
    df[
        [
            "Power_Norm",
            "Occupancy_Norm",
            "Temperature_Norm"
        ]
    ].head()
)

   Power_Norm  Occupancy_Norm  Temperature_Norm
0    0.399046        0.008197          0.530902
1    0.393613        0.008197          0.519081
2    0.361459        0.000000          0.515704
3    0.531655        0.000000          0.541371
4    0.241712        0.000000          0.516042


In [8]:
df["Power_State"] = (
    df["Power_Norm"]
    .round(1)
)

df["Occupancy_State"] = (
    df["Occupancy_Norm"]
    .round(1)
)

df["Temperature_State"] = (
    df["Temperature_Norm"]
    .round(1)
)

In [9]:
print(
    df[
        [
            "Power_State",
            "Occupancy_State",
            "Temperature_State"
        ]
    ].head()
)

   Power_State  Occupancy_State  Temperature_State
0          0.4              0.0                0.5
1          0.4              0.0                0.5
2          0.4              0.0                0.5
3          0.5              0.0                0.5
4          0.2              0.0                0.5


In [10]:
df["State_ID"] = (
    df["Power_State"].astype(str)
    + "-"
    + df["Occupancy_State"].astype(str)
    + "-"
    + df["Temperature_State"].astype(str)
)

print(
    df["State_ID"].head()
)

0    0.4-0.0-0.5
1    0.4-0.0-0.5
2    0.4-0.0-0.5
3    0.5-0.0-0.5
4    0.2-0.0-0.5
Name: State_ID, dtype: str


In [11]:
actions = {

    0: "Do Nothing",

    1: "Reduce HVAC",

    2: "Check Equipment",

    3: "Reduce Lighting"

}

actions

{0: 'Do Nothing', 1: 'Reduce HVAC', 2: 'Check Equipment', 3: 'Reduce Lighting'}

In [12]:
action_space = list(actions.keys())

print(action_space)

[0, 1, 2, 3]


In [13]:
num_actions = len(action_space)

print("Number of Actions:", num_actions)

Number of Actions: 4


In [14]:
print(actions)

{0: 'Do Nothing', 1: 'Reduce HVAC', 2: 'Check Equipment', 3: 'Reduce Lighting'}


In [15]:
action_id = 1

print(
    "Selected Action:",
    actions[action_id]
)

Selected Action: Reduce HVAC


In [16]:
action_df = pd.DataFrame({

    "Action_ID": actions.keys(),

    "Action_Name": actions.values()

})

action_df

,Action_ID,Action_Name
0,0,Do Nothing
1,1,Reduce HVAC
2,2,Check Equipment
3,3,Reduce Lighting


In [17]:
def get_next_state( #Environment model
    state,
    action
):

    power, occupancy, temperature = state

    if action == 0:

        # Do Nothing

        power = power

        temperature = temperature

    elif action == 1:

        # Reduce HVAC

        power = max(
            0.1,
            power - 0.3
        )

        temperature = min(
            1.0,
            temperature + 0.1
        )

    elif action == 2:

        # Check Equipment

        power = max(
            0.1,
            power - 0.1
        )

        temperature = temperature

    elif action == 3:

        # Reduce Lighting

        power = max(
            0.1,
            power - 0.2
        )

        temperature = min(
            1.0,
            temperature + 0.05
        )

    return (
        round(power, 1),
        occupancy,
        round(temperature, 1)
    )

In [18]:
print(
    get_next_state(
        (0.4,0.0,0.4),
        1
    )
)

(0.1, 0.0, 0.5)


In [19]:
def calculate_reward(
    state,
    next_state
):

    power_before = state[0]

    occupancy = state[1]

    temperature_before = state[2]

    power_after = next_state[0]

    temperature_after = next_state[2]

    energy_saving = (
        power_before
        - power_after
    )

    comfort_penalty = (
        occupancy
        *
        max(
            0,
            temperature_after
            - temperature_before
        )
    )

    reward = (
        energy_saving
        - comfort_penalty
    )

    return round(
        reward,
        2
    )

In [20]:
state=(0.4,0.0,0.4)

next_state=get_next_state(
    state,
    1
)

print(
    calculate_reward(
        state,
        next_state
    )
)

0.3


In [21]:
unique_states = sorted(
    df["State_ID"].unique()
)

state_to_index = {
    state: idx
    for idx, state in enumerate(unique_states)
}

index_to_state = {
    idx: state
    for state, idx in state_to_index.items()
}

print(
    list(state_to_index.items())[:10]
)

[('0.0-0.0-0.3', 0), ('0.0-0.0-0.7', 1), ('0.1-0.0-0.2', 2), ('0.1-0.0-0.3', 3), ('0.1-0.0-0.4', 4), ('0.1-0.0-0.5', 5), ('0.1-0.0-0.6', 6), ('0.1-0.0-0.9', 7), ('0.1-0.1-0.2', 8), ('0.1-0.1-0.3', 9)]


In [22]:
print(len(unique_states))

285


In [23]:
print(len(unique_states))
print(len(action_space))

285
4


In [24]:
'''experiences = []'''

'experiences = []'

In [25]:
'''state = (
    0.9,
    0.2,
    0.7
)'''

'state = (\n    0.9,\n    0.2,\n    0.7\n)'

In [26]:
'''for action in action_space:

    next_state = get_next_state(
        state,
        action
    )

    reward = calculate_reward(
        state,
        next_state
    )

    experience = {

        "state": state,

        "action": action,

        "reward": reward,

        "next_state": next_state

    }

    experiences.append(
        experience
    )'''

'for action in action_space:\n\n    next_state = get_next_state(\n        state,\n        action\n    )\n\n    reward = calculate_reward(\n        state,\n        next_state\n    )\n\n    experience = {\n\n        "state": state,\n\n        "action": action,\n\n        "reward": reward,\n\n        "next_state": next_state\n\n    }\n\n    experiences.append(\n        experience\n    )'

In [27]:
'''for exp in experiences:

    print(exp)'''

'for exp in experiences:\n\n    print(exp)'

In [28]:
#Create experience from all states
experiences = []

for state_id in unique_states:

    state_parts = state_id.split("-")

    current_state = (
        float(state_parts[0]),
        float(state_parts[1]),
        float(state_parts[2])
    )

    for action in action_space:

        next_state = get_next_state(
            current_state,
            action
        )

        reward = calculate_reward(
            current_state,
            next_state
        )

        experience = {

            "state": state_id,

            "action": actions[action],

            "reward": reward,

            "next_state": next_state

        }

        experiences.append(
            experience
        )

In [29]:
print(
    "Total Experiences:",
    len(experiences)
)

Total Experiences: 1140


In [30]:
#experiences of random 10 states
import random

for exp in random.sample(
    experiences,
    10
):

    print(exp)

{'state': '0.5-0.5-0.3', 'action': 'Reduce Lighting', 'reward': 0.2, 'next_state': (0.3, 0.5, 0.3)}
{'state': '0.4-0.2-0.4', 'action': 'Do Nothing', 'reward': 0.0, 'next_state': (0.4, 0.2, 0.4)}
{'state': '0.7-0.5-0.4', 'action': 'Check Equipment', 'reward': 0.1, 'next_state': (0.6, 0.5, 0.4)}
{'state': '0.3-0.6-0.3', 'action': 'Reduce HVAC', 'reward': 0.14, 'next_state': (0.1, 0.6, 0.4)}
{'state': '0.1-0.2-0.3', 'action': 'Reduce HVAC', 'reward': -0.02, 'next_state': (0.1, 0.2, 0.4)}
{'state': '0.2-0.2-0.5', 'action': 'Check Equipment', 'reward': 0.1, 'next_state': (0.1, 0.2, 0.5)}
{'state': '0.1-0.3-0.4', 'action': 'Reduce Lighting', 'reward': -0.03, 'next_state': (0.1, 0.3, 0.5)}
{'state': '0.5-0.5-0.4', 'action': 'Do Nothing', 'reward': 0.0, 'next_state': (0.5, 0.5, 0.4)}
{'state': '0.4-0.0-0.3', 'action': 'Reduce Lighting', 'reward': 0.2, 'next_state': (0.2, 0.0, 0.3)}
{'state': '0.6-0.1-0.3', 'action': 'Reduce Lighting', 'reward': 0.2, 'next_state': (0.4, 0.1, 0.3)}


In [31]:
#experience of first 10
for exp in experiences[:10]:

    print(exp)

{'state': '0.0-0.0-0.3', 'action': 'Do Nothing', 'reward': 0.0, 'next_state': (0.0, 0.0, 0.3)}
{'state': '0.0-0.0-0.3', 'action': 'Reduce HVAC', 'reward': -0.1, 'next_state': (0.1, 0.0, 0.4)}
{'state': '0.0-0.0-0.3', 'action': 'Check Equipment', 'reward': -0.1, 'next_state': (0.1, 0.0, 0.3)}
{'state': '0.0-0.0-0.3', 'action': 'Reduce Lighting', 'reward': -0.1, 'next_state': (0.1, 0.0, 0.3)}
{'state': '0.0-0.0-0.7', 'action': 'Do Nothing', 'reward': 0.0, 'next_state': (0.0, 0.0, 0.7)}
{'state': '0.0-0.0-0.7', 'action': 'Reduce HVAC', 'reward': -0.1, 'next_state': (0.1, 0.0, 0.8)}
{'state': '0.0-0.0-0.7', 'action': 'Check Equipment', 'reward': -0.1, 'next_state': (0.1, 0.0, 0.7)}
{'state': '0.0-0.0-0.7', 'action': 'Reduce Lighting', 'reward': -0.1, 'next_state': (0.1, 0.0, 0.8)}
{'state': '0.1-0.0-0.2', 'action': 'Do Nothing', 'reward': 0.0, 'next_state': (0.1, 0.0, 0.2)}
{'state': '0.1-0.0-0.2', 'action': 'Reduce HVAC', 'reward': 0.0, 'next_state': (0.1, 0.0, 0.3)}


In [32]:
experience_df = pd.DataFrame(
    experiences
)

experience_df

,state,action,reward,next_state
0,0.0-0.0-0.3,Do Nothing,0.00,"(0.0, 0.0, 0.3)"
1,0.0-0.0-0.3,Reduce HVAC,-0.10,"(0.1, 0.0, 0.4)"
2,0.0-0.0-0.3,Check Equipment,-0.10,"(0.1, 0.0, 0.3)"
3,0.0-0.0-0.3,Reduce Lighting,-0.10,"(0.1, 0.0, 0.3)"
4,0.0-0.0-0.7,Do Nothing,0.00,"(0.0, 0.0, 0.7)"
...,...,...,...,...
1135,0.9-0.1-0.6,Reduce Lighting,0.19,"(0.7, 0.1, 0.7)"
1136,1.0-0.0-0.7,Do Nothing,0.00,"(1.0, 0.0, 0.7)"
1137,1.0-0.0-0.7,Reduce HVAC,0.30,"(0.7, 0.0, 0.8)"
1138,1.0-0.0-0.7,Check Equipment,0.10,"(0.9, 0.0, 0.7)"


In [33]:
num_states = df["State_ID"].nunique()

num_actions = len(action_space)

q_table = np.zeros(
    (
        num_states,
        num_actions
    )
)

print(q_table.shape)

(285, 4)


In [34]:
q_table[:10]

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [35]:
print("Number of States:", len(unique_states))

print("Q Table Shape:", q_table.shape)

Number of States: 285
Q Table Shape: (285, 4)


In [36]:
#bellman equation
def update_q_value(
    state_idx,
    action,
    reward,
    next_state_idx
):

    old_value = q_table[
        state_idx,
        action
    ]

    next_max = np.max(
        q_table[
            next_state_idx
        ]
    )

    new_value = (
        old_value
        +
        alpha
        *
        (
            reward
            +
            gamma
            *
            next_max
            -
            old_value
        )
    )

    q_table[
        state_idx,
        action
    ] = new_value

In [37]:
alpha = 0.1

gamma = 0.9

episodes = 1000

In [38]:
q_table = np.zeros(
    (
        num_states,
        num_actions
    )
)

num_passes = 20

for _ in range(num_passes):

    for state in unique_states:

        state_idx = state_to_index[state]

        state_parts = state.split("-")

        current_state = (
            float(state_parts[0]),
            float(state_parts[1]),
            float(state_parts[2])
        )

        for action in action_space:

            next_state = get_next_state(
                current_state,
                action
            )

            reward = calculate_reward(
                current_state,
                next_state
            )

            next_state_id = (
                str(next_state[0])
                + "-"
                + str(next_state[1])
                + "-"
                + str(next_state[2])
            )

            if next_state_id in state_to_index:

                next_state_idx = state_to_index[
                    next_state_id
                ]

                update_q_value(
                    state_idx,
                    action,
                    reward,
                    next_state_idx
                )

In [39]:
print(
    q_table[:100]
)

[[ 0.         -0.08784233 -0.08784233 -0.08784233]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.         -0.00878423  0.          0.        ]
 [ 0.         -0.00878423  0.          0.        ]
 [ 0.         -0.00878423  0.         -0.00878423]
 [ 0.         -0.00878423  0.         -0.00878423]
 [ 0.          0.          0.          0.        ]
 [ 0.         -0.01756847  0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.         -0.0263527   0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.05474277  0.08784233  0.08784233  0.08784233]
 [ 0.05474277  0.08784233  0.08784233  0.08784233]
 [ 0.05474277  0.08784233  0.08

In [40]:
print("Min:", np.min(q_table))

print("Max:", np.max(q_table))

print("Mean:", np.mean(q_table))

Min: -0.08784233454094308
Max: 0.49660328138173965
Mean: 0.1264486247106595


In [41]:
def get_best_action(  #given a state , what action to take
    state_id
):

    state_idx = state_to_index[
        state_id
    ]

    best_action = np.argmax(
        q_table[
            state_idx
        ]
    )

    return best_action

In [42]:
sample_state = df[
    "State_ID"
].iloc[0]

print(
    get_best_action(
        sample_state
    )
)

1


In [43]:
def get_confidence_level(
    state_id
):

    state_idx = state_to_index[
        state_id
    ]

    q_values = q_table[
        state_idx
    ]

    best_q = np.max(
        q_values
    )

    total_q = np.sum(
        np.abs(q_values)
    )

    if total_q == 0:

        confidence = 0

    else:

        confidence = round(
            (
                best_q
                /
                total_q
            ) * 100,
            2
        )

    if confidence >= 70:

        level = "High"

    elif confidence >= 40:

        level = "Medium"

    else:

        level = "Low"

    return confidence, level

In [44]:
confidence, level = get_confidence_level(
    sample_state
)

print(
    "Confidence:",
    confidence,
    "%"
)

print(
    "Level:",
    level
)

Confidence: 30.54 %
Level: Low


In [45]:
q_df = pd.DataFrame(
    q_table,
    columns=actions
)

q_df

,0,1,2,3
0,0.000000,-0.087842,-0.087842,-0.087842
1,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...
280,0.274732,0.496603,0.365745,0.439538
281,0.249665,0.435050,0.333846,0.377984
282,0.210038,0.365528,0.304192,0.292381
283,0.241343,0.420548,0.312310,0.363483


In [46]:
policy_data = []

for state in unique_states:

    best_action = get_best_action(
        state
    )

    policy_data.append(
        [
            state,
            actions[best_action]
        ]
    )

policy_df = pd.DataFrame(
    policy_data,
    columns=[
        "State",
        "Best_Action"
    ]
)

print(policy_df.shape)

policy_df

(285, 2)


,State,Best_Action
0,0.0-0.0-0.3,Do Nothing
1,0.0-0.0-0.7,Do Nothing
2,0.1-0.0-0.2,Do Nothing
3,0.1-0.0-0.3,Do Nothing
4,0.1-0.0-0.4,Do Nothing
...,...,...
280,0.9-0.0-0.6,Reduce HVAC
281,0.9-0.0-0.7,Reduce HVAC
282,0.9-0.0-0.8,Reduce HVAC
283,0.9-0.1-0.6,Reduce HVAC


In [47]:
print(
    policy_df.shape
)

(285, 2)


In [48]:
action_id = get_best_action(
    sample_state
)

action_name = actions[
    action_id
]

print(action_name)

Reduce HVAC


In [49]:
state_tuple = tuple(
    map(
        float,
        sample_state.split("-")
    )
)

print(state_tuple)

(0.4, 0.0, 0.5)


In [50]:
action_id = get_best_action(
    sample_state
)

next_state = get_next_state(
    state_tuple,
    action_id
)

print(next_state)

(0.1, 0.0, 0.6)


In [51]:
power_before = state_tuple[0]

power_after = next_state[0]

projected_energy_saving = round(
    (
        (
            power_before
            - power_after
        )
        /
        power_before
    ) * 100,
    2
)

print(
    projected_energy_saving,
    "%"
)

75.0 %


In [52]:
print(sample_state)

print(next_state)

0.4-0.0-0.5
(0.1, 0.0, 0.6)


In [53]:
print(df.columns.tolist())

['date', 'Power Consumption', 'Outdoor Temperature', 'Occupancy', 'Power_Norm', 'Occupancy_Norm', 'Temperature_Norm', 'Power_State', 'Occupancy_State', 'Temperature_State', 'State_ID']


In [54]:
print("Best_Action" in df.columns)

print("Reward" in df.columns)

print("Projected_Energy_Saving" in df.columns)

print("Confidence" in df.columns)

False
False
False
False


In [60]:
results_df = pd.DataFrame(
    {
        "Current_State": [sample_state],

        "Recommended_Action": [action_name],

        "Predicted_Next_State": [
            str(next_state)
        ],

        "Reward": [reward],

        "Projected_Energy_Saving (%)": [
            projected_energy_saving
        ],

        "Confidence (%)": [
            confidence
        ],

        "Confidence_Level": [
            level
        ]
    }
)

results_df

,Current_State,Recommended_Action,Predicted_Next_State,Reward,Projected_Energy_Saving (%),Confidence (%),Confidence_Level
0,0.4-0.0-0.5,Reduce HVAC,"(0.1, 0.0, 0.6)",0.2,75.0,30.54,Low


In [66]:
results_df.to_csv(
    "../data/decision_results.csv",
    index=False
)

print(
    "decision_results.csv saved successfully!"
)

decision_results.csv saved successfully!


In [75]:
print("get_best_action" in globals())
print("get_next_state" in globals())
print("calculate_reward" in globals())
print("get_confidence_level" in globals())

True
True
True
True


In [76]:
def state_id_to_tuple(state_id):

    power, occupancy, temperature = map(
        float,
        state_id.split("-")
    )

    return (
        power,
        occupancy,
        temperature
    )

In [83]:
policy_results = []

for state_id in unique_states:

    # Convert string State_ID to tuple

    state = state_id_to_tuple(
        state_id
    )

    # Best Action

    best_action = get_best_action(
        state_id
    )

    action_name = actions[
        best_action
    ]

    # Predicted Next State

    next_state = get_next_state(
        state,
        best_action
    )

    # Reward

    reward = calculate_reward(
        state,
        next_state
    )

    # Confidence

    confidence, confidence_level = (
        get_confidence_level(
            state_id
        )
    )

    # Projected Energy Saving

    current_power = state[0]

    next_power = next_state[0]

    if current_power > 0:

        projected_saving = round(
            (
                (
                    current_power
                    - next_power
                )
                /
                current_power
            )
            * 100,
            2
        )

    else:

        projected_saving = 0

    # Store Results

    policy_results.append(
        {
            "Current_State":
                state_id,

            "Recommended_Action":
                action_name,

            "Predicted_Next_State":
                str(next_state),

            "Reward":
                reward,

            "Projected_Energy_Saving (%)":
                projected_saving,

            "Confidence (%)":
                confidence,

            "Confidence_Level":
                confidence_level
        }
    )

policy_results_df = pd.DataFrame(
    policy_results
)

policy_results_df.head(100000)

,Current_State,Recommended_Action,Predicted_Next_State,Reward,Projected_Energy_Saving (%),Confidence (%),Confidence_Level
0,0.0-0.0-0.3,Do Nothing,"(0.0, 0.0, 0.3)",0.00,0.00,0.00,Low
1,0.0-0.0-0.7,Do Nothing,"(0.0, 0.0, 0.7)",0.00,0.00,0.00,Low
2,0.1-0.0-0.2,Do Nothing,"(0.1, 0.0, 0.2)",0.00,0.00,0.00,Low
3,0.1-0.0-0.3,Do Nothing,"(0.1, 0.0, 0.3)",0.00,0.00,0.00,Low
4,0.1-0.0-0.4,Do Nothing,"(0.1, 0.0, 0.4)",0.00,0.00,0.00,Low
...,...,...,...,...,...,...,...
280,0.9-0.0-0.6,Reduce HVAC,"(0.6, 0.0, 0.7)",0.30,33.33,31.50,Low
281,0.9-0.0-0.7,Reduce HVAC,"(0.6, 0.0, 0.8)",0.30,33.33,31.15,Low
282,0.9-0.0-0.8,Reduce HVAC,"(0.6, 0.0, 0.9)",0.30,33.33,31.18,Low
283,0.9-0.1-0.6,Reduce HVAC,"(0.6, 0.1, 0.7)",0.29,33.33,31.44,Low


In [84]:
print(get_best_action)
print(get_confidence_level)
print(calculate_reward)

<function get_best_action at 0x000001A13A97CA40>
<function get_confidence_level at 0x000001A13A97FC40>
<function calculate_reward at 0x000001A1224C5EE0>


In [86]:
print(
    len(policy_results_df)
)

285


In [88]:
policy_results_df.to_csv(
    "../data/policy_results.csv",
    index=False
)

print(
    "policy_results.csv saved successfully!"
)

policy_results.csv saved successfully!


In [90]:
print(policy_results_df.columns)

Index(['Current_State', 'Recommended_Action', 'Predicted_Next_State', 'Reward',
       'Projected_Energy_Saving (%)', 'Confidence (%)', 'Confidence_Level'],
      dtype='str')
